In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── 1. Load Real Model & Tokenizer ───────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-0.5B"
print(f"Loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    output_hidden_states=True,
    output_attentions=True
)
model.eval()

# ── 2. Setup the Prompt & Candidates ─────────────────────────────────────────
STORY = (
    "There was a quiet house in the city. Several people lived there "
    "including John, Michael, and Sarah. One night a terrible crime "
    "occurred in the house. The police investigated every suspect carefully. "
    "After reviewing the evidence they concluded that the murderer was"
)

CANDIDATES = ["John", "Michael", "Sarah"]
CORRECT    = "John"

inputs = tokenizer(STORY, return_tensors="pt").to(model.device)
SEQ_LEN = inputs.input_ids.shape[1]

# Get candidate token IDs (handling Qwen's spacing)
cand_ids = {name: tokenizer.encode(" " + name, add_special_tokens=False)[0] for name in CANDIDATES}

# ── 3. Forward Pass ──────────────────────────────────────────────────────────
with torch.no_grad():
    outputs = model(**inputs)

raw_hidden_states = outputs.hidden_states
N_LAYERS = len(raw_hidden_states) - 1

# Extract the hidden state for the *last token* across all layers
final_hidden = torch.stack([h[0, -1, :] for h in raw_hidden_states]).cpu().float().numpy()
D_MODEL = final_hidden.shape[1]

# Get the unembedding weights (lm_head)
W_U = model.lm_head.weight.detach().cpu().float().numpy()

# ── 4. Compute Cosine Similarity ─────────────────────────────────────────────
sims = {}
norms_hidden = np.linalg.norm(final_hidden, axis=1) + 1e-8

for name in CANDIDATES:
    target_vec = W_U[cand_ids[name]]
    norm_target = np.linalg.norm(target_vec) + 1e-8
    raw_dot = final_hidden @ target_vec
    sims[name] = raw_dot / (norms_hidden * norm_target)

# ── 5. PCA Projection ────────────────────────────────────────────────────────
def pca_3d(matrix):
    C = matrix - matrix.mean(0)
    cov = C.T @ C / (len(C) - 1)
    vals, vecs = np.linalg.eigh(cov)
    idx  = np.argsort(vals)[::-1]
    return C @ vecs[:, idx[:3]], vecs[:, idx[:3]]

all_vecs = np.vstack([final_hidden, np.stack([W_U[cand_ids[n]] for n in CANDIDATES], axis=0)])
projected, _ = pca_3d(all_vecs)
traj_3d = projected[:N_LAYERS + 1]
cand_3d = projected[N_LAYERS + 1:]

# ── 6. Plotting (Inline for Colab) ───────────────────────────────────────────
PALETTE = {"John": "#e63946", "Michael": "#457b9d", "Sarah": "#2a9d8f"}
LINESTYLE = {"John": "-", "Michael": "--", "Sarah": "-."}
layers = np.arange(N_LAYERS + 1)

# Plot 1: Cosine Similarity
fig1, ax = plt.subplots(figsize=(10, 5))
fig1.patch.set_facecolor("#0d1117")
ax.set_facecolor("#161b22")

for name in CANDIDATES:
    ax.plot(layers, sims[name], color=PALETTE[name], linestyle=LINESTYLE[name],
            linewidth=2.5 if name == CORRECT else 1.8, marker="o", label=name)

ax.axhline(0, color="#444", linewidth=0.8, linestyle=":")
ax.set_title("Qwen 2.5 (0.5B): Hidden-state alignment with candidate logits", color="white", fontsize=14)
ax.set_xlabel("Layer index", color="white"); ax.set_ylabel("Cosine similarity", color="white")
ax.tick_params(colors="white"); ax.legend(facecolor="#1e2530", labelcolor="white")
for sp in ax.spines.values(): sp.set_edgecolor("#333")
plt.show()

# Plot 2: 3D PCA Trajectory
fig2 = plt.figure(figsize=(10, 8))
fig2.patch.set_facecolor("#0d1117")
ax3 = fig2.add_subplot(111, projection="3d")
ax3.set_facecolor("#161b22")

cmap_traj = plt.cm.plasma
for i in range(N_LAYERS):
    ax3.plot(traj_3d[i:i+2, 0], traj_3d[i:i+2, 1], traj_3d[i:i+2, 2], color=cmap_traj(i/N_LAYERS), linewidth=2.2)

ax3.scatter(traj_3d[:, 0], traj_3d[:, 1], traj_3d[:, 2], c=np.arange(N_LAYERS + 1), cmap="plasma", s=40)
ax3.text(traj_3d[0, 0], traj_3d[0, 1], traj_3d[0, 2], "  x⁽⁰⁾", color="white")
ax3.text(traj_3d[-1, 0], traj_3d[-1, 1], traj_3d[-1, 2], f"  x⁽{N_LAYERS}⁾", color="white")

for j, name in enumerate(CANDIDATES):
    ax3.scatter(cand_3d[j, 0], cand_3d[j, 1], cand_3d[j, 2], color=PALETTE[name], s=150,
                marker="*" if name == CORRECT else "s", edgecolors="white")
    ax3.text(cand_3d[j, 0], cand_3d[j, 1], cand_3d[j, 2], f"  w_{name}", color=PALETTE[name])

ax3.set_title("PCA 3-D trajectory of Qwen's final-token hidden state", color="white")
ax3.tick_params(colors="#888")
for pane in [ax3.xaxis.pane, ax3.yaxis.pane, ax3.zaxis.pane]: pane.fill = False; pane.set_edgecolor("#333")
plt.show()

Loading Qwen/Qwen2.5-0.5B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['output_attentions', 'output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]